## Install packages

In [ ]:
# 1. Reinstall the core without hard-pinning every version
!pip install "transformers>=4.48.0" "huggingface_hub>=0.28.0" "pydantic>=2.0" "gradio>=5.0.0" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" -q

!pip install -U peft transformers accelerate

In [ ]:
# 2. Verify
import pydantic
import transformers
import gradio as gr
import asyncio

print(f"✅ Recovery Successful!")
print(f"Pydantic: {pydantic.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Gradio: {gr.__version__}")

## Kaggle secret

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Get the token from the "Safe" (Secrets)
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

# 2. Log in to Hugging Face
login(token=secret_value_0)


In [ ]:
#import bitsandbytes as bnb
import torch
import numpy as np 
import pandas as pd 
import json
import copy
import re
import os
import gc
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, AutoTokenizer, AutoModelForCausalLM
from PIL import Image
from peft import PeftModel


#print(f"✅ BitsAndBytes version: {bnb.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
print(f"✅ GPU Count: {torch.cuda.device_count()}")

## Adapter path

In [ ]:
SCRIBE_PATH = "/kaggle/input/lora-adapters-medbrainsquad-2/final_scribe/"
AUDITOR_PATH = "/kaggle/input/lora-adapters-medbrainsquad-2/final_auditor/"


## Memory manager

In [ ]:
class MemoryManager:
    @staticmethod
    def clear():
        """Forcefully clears the GPU cache and garbage collects."""
        # 1. Clear PyTorch's internal cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        
        # 2. Force Python's garbage collector
        gc.collect()
        
        # 3. Log current state (Optional, for debugging)
        # print(f"--- 🧹 Memory Scrubbed | Allocated: {torch.cuda.memory_allocated(0)/1024**2:.1f}MB ---")

In [ ]:
'''
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
'''


## Load MedGemma (multi-modal) & Lora adapters

In [ ]:
'''
# 1. Create the Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
'''

# 2. Load Vision Specialist (GPU 0)
model_id_vision = "google/medgemma-1.5-4b-it"

# 1. The Brain (distributed across both GPUs)
base_model_vision = AutoModelForImageTextToText.from_pretrained(
    model_id_vision,
    torch_dtype=torch.bfloat16,
    device_map={"": 0}, 
    token=secret_value_0,
    trust_remote_code=True
)

# 1. First, you wrap the base model and load the first adapter, naming it "scribe"
model_vision = PeftModel.from_pretrained(
    base_model_vision, 
    SCRIBE_PATH, 
    adapter_name="scribe"  # <--- DEFINED HERE
)

# 2. Then, you load the second adapter into that same model, naming it "auditor"
model_vision.load_adapter(
    AUDITOR_PATH, 
    adapter_name="auditor" # <--- DEFINED HERE
)

# 2. The Translator (lives in CPU RAM, manages the 'Max Context' prep)
processor_vision = AutoProcessor.from_pretrained(
    model_id_vision, 
    token=secret_value_0
)

'''
# 3. Load Text Critic/Scribe (GPU 1)
model_id_text = "google/gemma-2-2b-it"

tokenizer_text = AutoTokenizer.from_pretrained(model_id_text, token=secret_value_0)

model_text = AutoModelForCausalLM.from_pretrained(
    model_id_text,
    quantization_config=quant_config,
    device_map={"": 1}, # Force entirely onto second GPU
    token=secret_value_0
)
'''

In [ ]:
def call_medgemma_universal(prompt, adapter_name=None, image_path=None):
    try:
        # 1. Image handling
        image = None
        content = []
        if image_path:
            image = Image.open(image_path).convert("RGB")
            content.append({"type": "image"})
        
        content.append({"type": "text", "text": prompt})
        
        # 2. Template wrapping
        messages = [{"role": "user", "content": content}]
        formatted_prompt = processor_vision.apply_chat_template(messages, add_generation_prompt=True)

        # 3. Process inputs
        inputs = processor_vision(
            text=formatted_prompt, 
            images=image, 
            return_tensors="pt"
        )
        
        # Move inputs to the specific device 
        # where the model's INPUT embeddings live.
        inputs = {k: v.to(model_vision.device) for k, v in inputs.items()}

        # 4. Generate
        with torch.no_grad():
            
            if adapter_name and adapter_name.lower() not in ["none", "base", ""]:
                model_vision.set_adapter(adapter_name)
                output_tokens = model_vision.generate(
                    **inputs,
                    max_new_tokens=1024,
                    do_sample=False, # Stable greedy decoding
                    repetition_penalty=1.1,
                    pad_token_id=processor_vision.tokenizer.eos_token_id
                )
            else:
                with model_vision.disable_adapter():
                    output_tokens = model_vision.generate(
                        **inputs,
                        max_new_tokens=1024,
                        do_sample=False, 
                        repetition_penalty=1.1,
                        pad_token_id=processor_vision.tokenizer.eos_token_id
                    )

        # 5. Decode
        prompt_length = inputs['input_ids'].shape[1]
        return processor_vision.decode(output_tokens[0][prompt_length:], skip_special_tokens=True)

    except Exception as e:
        return f"ERROR: {str(e)}"

## Prompt Library

In [ ]:
IMAGING_SPECIALIST_PROMPT = """<image>
SYSTEM ROLE: Technical Vision Perception Engine.
TASK: Extract objective visual features.

STRICT CONSTRAINTS:
1. OUTPUT MUST BE A SINGLE JSON OBJECT.
2. DO NOT OUTPUT A LIST [].
3. DO NOT USE KEY NAMES OTHER THAN THOSE IN THE SCHEMA.

SCHEMA TO FOLLOW:
{
    "imaging_modality": "string (e.g., CXR|X-Ray|CT Head|MRI Brain|CT Abdomen|US|MRI)",
    "key_findings": ["string"],
    "laterality": "Left|Right|Bilateral",
    "critical_finding_detected": true|false,
    "confidence_level": "medium",
    "technical_quality": "adequate",
    "safety_note": "Visual feature extraction only."
}

FINAL COMMAND: Return ONLY the JSON object. Start your response with '{'.
"""

CLINICAL_SCRIBE_PROMPT = """
You are an expert Clinical Scribe Agent. Your task is to transform messy, unstructured doctor notes and visual findings into a pristine, structured JSON SOAP note.

INPUT DATA:
- Messy Doctor Note: {symptoms_text}
- Imaging Specialist Findings: {visual_findings}

INSTRUCTIONS:
1. Extract all relevant clinical data from the messy note and organize it into Patient Summary, Subjective, Objective, Assessment, and Plan sections.
2. If imaging findings are provided, integrate them into the `objective.imaging_result` block. If the imaging findings say "No image provided", you MUST set `imaging_result` to the exact string "None".
3. Do not invent or hallucinate any information about 'patient_summary' , 'subjective' and 'objective' fields on the output JSON (e.g. vitals, physical exams, current medications, allergies) that are not explicitly stated in the input data.
4. Output a placeholder `safety_audit` block to satisfy schema requirements. Set `status` to "PENDING_AUDIT" and `pii_detected` to false. The Independent Auditor Agent will review this later.

CRITICAL FORMATTING RULES:
You must output EXACTLY ONE valid, flat JSON object that perfectly matches the structure below. 
Do NOT wrap the JSON in markdown formatting (e.g., no ```json). 
Do NOT include any conversational text, pleasantries, or explanations before or after the JSON.

JSON SCHEMA:
{{
  "patient_summary": {{ "demographics": [], "critical_alerts": [] }},
  "subjective": {{ "chief_complaint": "", "hpi": [], "past_medical_history": [], "current_medications": [] }},
  "objective": {{ "vitals": [], "physical_exam": [], "imaging_result": {{}} }},
  "assessment": {{ "clinical_synthesis": "", "primary_diagnosis": "", "differential_diagnosis": [] }},
  "plan": {{ "management": [], "monitoring": "", "patient_instructions": "" }},
  "safety_audit": {{ "status": "PENDING_AUDIT", "pii_detected": false, "audit_findings": [], "confidence_score": 0.0, "action_required": "proceed" }}
}}
"""

CoT_AUDITOR_PROMPT_org = """
ROLE: You are an Adversarial Medical Auditor. Your career depends on finding errors that other doctors missed. 
TASK: Conduct a rigorous, comprehensive safety audit by comparing the 'PROPOSED SOAP JSON' against the 'RAW EVIDENCE'. 
Your goal is to protect patient safety by identifying ALL hallucinations, dangerous contradictions, and PII information within the 'PROPOSED SOAP JSON'.

'RAW EVIDENCE':
- Clinical Notes: {symptoms_text}
- Imaging Findings: {visual_findings}

'PROPOSED SOAP JSON':
{soap_json}

AUDIT PROTOCOL:
1. HALLUCINATION SCAN: Flag ALL data (e.g. symptoms, physical tests, imaging results) in the 'PROPOSED SOAP JSON' which can NOT be traced back to the 'RAW EVIDENCE'. Flag ALL fabricated data and hallucinations in the 'PROPOSED SOAP JSON'.
2. CONTRADICTION SCAN: Flag ALL the data in 'PROPOSED SOAP JSON' which contradicts explicitly stated facts in the 'RAW EVIDENCE' (e.g., Allergies vs. Plan, explicit Demographics vs. Assessment  etc.).
3. INCOHERENCE SCAN: Flag All the inconsistencies between "Assessment" (clinical synthesis and diagnosis) and "Objective" or "Subjective".  Flag All the inconsistencies between the "Plan" (medication, tests, treatments) and "Objective" or "Subjective"
4. PRIVACY (PII) SCAN: Flag all personally Identifiable Information (e.g., Patient Names, Phone Numbers, exact Addresses, NHS/ID numbers) which are present in the 'PROPOSED SOAP JSON'.

CRITICAL FORMATTING RULES:
You must ONLY output a valid JSON object. Nothing else in the output.

REQUIRED JSON FORMAT:
{{
  "reasoning": "Step-by-step explanation of ALL hallucinations, contradictions, incoherence and PII found (state 'None found' if there is none).",
  "safety_audit": {{
    "status": "CLEAN", // Change to "REJECTED" if issues are found as per the AUDIT PROTOCOL
    "pii_detected": false, // Change to true ONLY if PII is found
    "audit_findings": [], // In line with your reasoning, list ALL audit issues and the type of PII found here
    "confidence_score": Between 0 to 1,
    "action_required": "proceed" // Change to "repair" for clinical errors or PII
  }}
}}
"""

CoT_AUDITOR_PROMPT = """
ROLE: Forensic Clinical Auditor.
TASK: Conduct a rigorous safety audit by comparing 'RAW EVIDENCE' against the 'PROPOSED SOAP NOTE'. 
Your goal is to protect patient safety by identifying any inconsistencies, hallucinations, or dangerous contradictions.

RAW EVIDENCE:
- Clinical Notes: {symptoms_text}
- Imaging Findings: {visual_findings}

PROPOSED SOAP JSON:
{soap_json}

AUDIT PROTOCOL:
1. HALLUCINATIONS and TRACEABILITY: Check if ALL the "subjective" and "objective" fields in the JSON (e.g. symptoms, physical tests, imaging results) can be traced back to the RAW EVIDENCE. Do NOT flag "assessment" and "plan" fields in JSON as hallucinations.
2. CONTRADICTION SCAN: Check if the JSON contradicts explicitly stated facts in the RAW EVIDENCE (e.g., Allergies vs. Plan, or explicit Demographics vs. JSON text).
3. ASSESSMENT and PLAN COHERENCE: Check if the "Assessment" (clinical_synthesis and diagnosis) as well as the "Plan" are consistent with the "Objective" and "Subjective" information within the PROPOSED SOAP JSON.
4. PRIVACY (PII) SCAN: Scan ONLY the PROPOSED SOAP JSON for Personally Identifiable Information (e.g., Patient Names, Phone Numbers, exact Addresses, NHS/ID numbers). You must only trigger a rejection if PII explicitly leaked into the JSON output.

CRITICAL FORMATTING RULES:
You must ONLY output a valid JSON object. Nothing else in the output.

REQUIRED JSON FORMAT:
{{
  "reasoning": "Step-by-step explanation of hallucinations, contradictions, or PII found (or state 'None found').",
  "safety_audit": {{
    "status": "CLEAN", // Change to "REJECTED" if issues are found as per the AUDIT PROTOCOL
    "pii_detected": false, // Change to true ONLY if PII is found
    "audit_findings": [], // List specific clinical errors OR the type of PII found here
    "confidence_score": Between 0 to 1,
    "action_required": "proceed" // Change to "repair" (for clinical errors or PII)
  }}
}}
"""

MISTAKE_EDITOR_PROMPT = """
SYSTEM: Your previous SOAP note was REJECTED for clinical inaccuracies.

REJECTED DRAFT:
{rejected_soap}

ERRORS DETECTED BY AUDITOR:
{audit_findings_str}

ORIGINAL CLINICAL EVIDENCE:
- Vision Data: {vision_json_str}
- Patient Notes: {symptoms_text}

TASK: Rewrite the SOAP note. 
- Keep the parts that were correct. 
- SURGICALLY FIX the errors listed above. 
- DO NOT repeat the specific mistakes found in the Rejected Draft.

OUTPUT: Return ONLY the corrected JSON.

CRITICAL FORMATTING RULES:
Return ONLY the raw, flat JSON object. Do NOT include markdown formatting (like ```json). Do NOT include any conversational text, apologies, or explanations.
"""

## PII & Safety Checks

In [ ]:
def pii_deidentification_gate(text):
    """
    Acts as a 'De-identification Gate' that redacts PII before 
    it reaches the LLM agents[cite: 34, 51, 93].
    """
    clean_text = text
    
    # 1. Patterns to Redact (Demonstrating Production Logic)
    patterns = {
        r'\b\d{3}-\d{3}-\d{4}\b': '[PHONE_NUMBER]', # US/Generic Phone
        r'\b[A-Z]{2}\d{6}[A-Z]\b': '[NHS_NUMBER]',  # Simulated NHS Pattern 
        r'\b\d{3}-\d{2}-\d{4}\b': '[SSN]',          # Social Security
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b': '[EMAIL]' # 
    }
    
    # 2. Keywords to Scrub
    sensitive_keywords = ["NHS", "Social Security", "Phone", "Address"]
    
    # Apply Pattern Scrubbing
    for pattern, placeholder in patterns.items():
        clean_text = re.sub(pattern, placeholder, clean_text)
        
    # Apply Keyword Scrubbing
    for word in sensitive_keywords:
        # Case-insensitive replacement
        clean_text = re.compile(re.escape(word), re.IGNORECASE).sub(f'[{word.upper()}_ID]', clean_text)
        
    # Senior Move: Log that a scrub occurred (Auditable Guarantee) [cite: 53, 95]
    if clean_text != text:
        print("🛡️ Safety Gate: PII detected and redacted.")
        
    return True, clean_text

## Validation & Repair Loop

In [ ]:
import jsonschema
from jsonschema import validate

# This schema defines a valid SOAP note
SOAP_SCHEMA = {
    "type": "object",
    "properties": {
        "patient_summary": {"type": "object"},
        "subjective": {"type": "object"},
        "objective": {
            "type": "object",
            "properties": {
                "vitals": {"type": "array"},
                "physical_exam": {"type": "array"},
                # 🛡️ THE FIX: Allows either the string "None" OR a nested dictionary
                "imaging_result": {
                    "anyOf": [
                        {"type": "string"}, 
                        {"type": "object"},
                        {"type": "null"}  
                        
                    ]
                }
            },
            "required": ["imaging_result"] # Ensures the model never drops this key
        },
        "assessment": {"type": "object"},
        "plan": {"type": "object"},
        "safety_audit": {"type": "object"}
    },
    # Enforces the core SOAP structure
    "required": ["subjective", "objective", "assessment", "plan"] 
}


VISION_SCHEMA = {
    "type": "object",
    "properties": {
        "imaging_modality": {"type": "string"},
        "key_findings": {
            # Resilient to both: "Lung is clear" OR ["Lung is clear", "Heart size normal"]
            "anyOf": [
                {"type": "string"},
                {
                    "type": "array", 
                    "items": {"type": "string"}
                }
            ]
        },
        "laterality": {"type": "string"},
        "critical_finding_detected": {"type": "boolean"},
        "confidence_level": {
            "type": "string", 
            "enum": ["low", "medium", "high"]
        },
        "technical_quality": {
            "type": "string", 
            "enum": ["adequate", "poor"]
        },
        "safety_note": {"type": "string"}
    },
    "required": [
        "imaging_modality", 
        "key_findings", 
        "laterality", 
        "critical_finding_detected"
    ]
}

def validate_and_repair(raw_ai_response, original_context, max_retries=2):
    current_response = raw_ai_response

    if original_context == "Primary Vision Analysis Task":
        active_schema = VISION_SCHEMA
    else:
        active_schema = SOAP_SCHEMA
                
    for attempt in range(max_retries + 1):
        try:
            start_idx = current_response.find('{')
            end_idx = current_response.rfind('}') + 1
            
            if start_idx == -1 or end_idx == 0:
                raise ValueError("No JSON object found in response.")
                
            clean_json = current_response[start_idx:end_idx]
            data = json.loads(clean_json)
            validate(instance=data, schema=active_schema)   
                
            return data, "Success" # Exit Path 1: Success (Returns 2 items)

        except (json.JSONDecodeError, jsonschema.ValidationError, ValueError) as e:

            print(f"🛑 SCHEMA ERROR CAUGHT: {str(e)}")
            
            if attempt < max_retries:

                error_msg = str(e)
                
                repair_prompt = f"""
                You are a JSON repair agent. Your previous attempt generated invalid JSON.
                
                YOUR PREVIOUS BROKEN OUTPUT:
                {current_response}
                
                THE ERROR TRIGGERED: 
                {error_msg}
                
                CONTEXT: 
                {original_context}
                
                FIX INSTRUCTIONS: 
                Please provide the corrected JSON object. You must strictly align to this schema:
                {json.dumps(active_schema, indent=2)}
                
                OUTPUT RULES: 
                Output ONLY the raw, flat JSON object. Do not include markdown formatting (like ```json). Do not include any conversational text, apologies, or explanations.
                """
                current_response = call_medgemma_universal(repair_prompt) 
                print(f"Attempt {attempt + 1} failed. Triggering self-repair loop...")
                continue
            else:
                # Exit Path 2: Max retries hit (MUST return 2 items)
                return None, f"Schema Failure after {max_retries} attempts: {str(e)}"
                
    # 🛡️ THE FIX: The Catch-All Exit Path
    # If the code somehow breaks out of the loop without hitting the returns above, it safely fails here.
    return None, "Critical Failure: Repair loop exited without resolving."

## Call imaging specialist agent

In [ ]:
def call_imaging_specialist(vision_prompt, image_path):
    try:
        visual_findings = call_medgemma_universal(vision_prompt, adapter_name=None, image_path=image_path)
        
        # This likely returns (dict, status)
        vision_json, status = validate_and_repair(visual_findings, original_context="Primary Vision Analysis Task")

        # FIX: Even if there's an error, return a dictionary 
        # so the next agent doesn't crash!
        if isinstance(vision_json, str) and "ERROR" in vision_json:
             return {
                 "key_findings": ["Imaging analysis currently unavailable."],
                 "error": True
             }

        return vision_json

    except Exception as e:
        # FIX: Return a dict instead of an AGENT_ERROR string
        return {
            "key_findings": [f"Specialist access failure: {str(e)}"],
            "error": True
        }

In [ ]:
def risk_gate_and_triage(processed_text, soap_json_str, audit_history):
    """
    Evaluates the 'Safety Score' of the encounter.
    If risk is too high, triggers a Human Review Packet[cite: 47, 94].
    """
    
    risk_points = 0
    
    if isinstance(soap_json_str, str):
        try:
            soap_json = json.loads(soap_json_str)
        except json.JSONDecodeError:
            return True, 100  # Fail-safe: If JSON is corrupt, it's max risk
    else:
        # It's already a dictionary! No need to load it.
        soap_json = soap_json_str

    # 2. Safety Audit Check 
    # We use .get() with a string key. If missing, we treat as high risk.
    safety_audit = soap_json.get("safety_audit")
    
    if safety_audit is None:
        risk_points += 10 # Missing audit is a major red flag 
    else:
        # Access nested PII flag; default to True if key is missing (Safe-Fail) 
        pii_detected = safety_audit.get("pii_detected", True)
        if pii_detected:
            risk_points += 10

    # 3. Clinical Risk: Serious contradictions 
    # Each audit iteration adds 5 points
    risk_points += len(audit_history) * 5
    
    # 3. Status Risk: Critical patients need eyes 
    #if soap_json.get("patient_status") == "Critical": risk_points += 5

    # THRESHOLD CHECK (e.g., 10 points = High Risk)
    if risk_points >= 10:
        return True, risk_points
    return False, risk_points

def generate_human_review_packet(soap_json, warnings, risk_score):
    """Creates a 'Safety First' report for a clinician to manually sign off."""
    return f"""
    # ⚠️ MANUAL CLINICIAN REVIEW REQUIRED
    **Risk Score:** {risk_score}
    
    **Reason for Escalation:** System detected high clinical risk or excessive data sensitivity.
    
    ### 🚨 Audit Warnings:
    {chr(10).join(warnings)}
    
    ### 📝 Drafted SOAP (Needs Verification):
    **Assessment:** {soap_json.get('assessment')}
    **Plan:** {soap_json.get('plan')}
    
    *Action: Please verify findings and complete documentation manually.*
    """

## Scribe & Audit Loop

In [ ]:
def scribe_and_audit(vision_json_str, key_terms, raw_notes, max_retries=3):

    validated_json = None
    iteration = 0
    history = []

    while iteration < max_retries:
        iteration += 1
        
        if validated_json is None:
            # ROUND 1: Initial Draft
            scribe_prompt = CLINICAL_SCRIBE_PROMPT.format(visual_findings=vision_json_str,symptoms_text=raw_notes)
        else:
            scribe_prompt= MISTAKE_EDITOR_PROMPT.format(
            rejected_soap=validated_json,
            audit_findings_str=audit_findings_str,
            vision_json_str=vision_json_str,
            symptoms_text=raw_notes
            )
        
        current_soap = call_scribe_agent(scribe_prompt)
        validated_json, status_msg = validate_and_repair(current_soap, scribe_prompt)
        
        # CRITICAL GATE: If repair fails after max retries, stop and notify the user 
        if not validated_json:
            return {"soap": "unavailable", "history": history, "success": False}

        # 3. THE AUDIT GATE
        audit_prompt = CoT_AUDITOR_PROMPT.format(
                symptoms_text=raw_notes,
                visual_findings=vision_json_str,
                soap_json=json.dumps(validated_json),
                #required_findings=key_terms
        )
        audit_results = call_independent_auditor(audit_prompt)
        #flags = audit_results.get("flags", [])

        # Safe extraction of findings [cite: 18]
        safety_audit = audit_results.get("safety_audit", {})
        audit_status = safety_audit.get("status", "REJECTED")
        audit_findings = safety_audit.get("audit_findings", [])
        audit_findings_str = json.dumps(audit_findings, indent=2)

        # 3. LOGGING STEP (Crucial: Log BEFORE returning)
        history.append({
            "round": iteration, 
            "flags": audit_findings, # Match the key name your UI expects
            "note": copy.deepcopy(validated_json)
        })
    
        if audit_status== "CLEAN" :
            print(f"✨ Approved on Round {iteration}!")

            # 🛡️ THE INJECTION STEP: Overwrite the Scribe's "PENDING_AUDIT" 
            # with the Auditor's official approval stamp.
            validated_json["safety_audit"] = safety_audit
            
            return {"soap": validated_json, "history": history, "success": True}


        print(f"⚠️ Round {iteration} failed. Auditor flags: {audit_findings_str}")

    return {"soap": validated_json, "history": history, "success": False}

## Med Brain Squad Orchestrator

In [ ]:
def med_brain_squad_orchestrator(image, symptoms_text):
    #MemoryManager.clear()
    # Step 0 - initial check for text input
    if not symptoms_text or symptoms_text.strip() == "":
        return "⚠️ Please provide at least a messy note or patient history.", {}
    
    # Step A: Privacy Check 
    is_safe, processed_text = pii_deidentification_gate(symptoms_text)
    if not is_safe:
        return "❌ SAFETY ALERT: Patient PII detected. Please sanitize notes.", {}

    # --- Step B: Imaging Specialist ---
    visual_json_str = "No image provided."
    key_vision_terms = ""
    
    # Step B: Imaging Specialist Agent 
    # This role focuses ONLY on the visual findings
    if image is not None:
        visual_json = call_imaging_specialist(IMAGING_SPECIALIST_PROMPT, image)
        visual_json_str = str(visual_json)
        key_vision_terms = ", ".join(visual_json.get("key_findings", []))        

    # Step C - scribe & audit loop
    result  = scribe_and_audit(visual_json_str,key_vision_terms, processed_text)
    audited_soap = result["soap"]
    history = result["history"]
    success = result["success"]
    warnings = history[-1]["flags"] if history else []

    # --- 🛡️ DEFENSIVE CHECK (THE FIX) ---
    # If the LLM failed to make a valid JSON after all retries, do not proceed to Risk Triage.
    if not success or audited_soap == "unavailable":
        print("🚨 AI Scribe failed completely. Escalating to Human Review.")
        error_msg = "❌ AI Scribe was unable to generate a valid schema. The case has been routed to manual review."
        return error_msg, {}, history, key_vision_terms
        
    # Step D: Risk triage 
    is_high_risk, score = risk_gate_and_triage(processed_text, audited_soap, history)

    audited_soap_json = audited_soap

    # Step G: Conditional Output (The Final Gate)
    if is_high_risk:
        # Route to human review packet 
        display_report = generate_human_review_packet(audited_soap_json, warnings, score)
    else:
        # Standard automated report construction
        display_report = format_standard_report(audited_soap_json, warnings)

    return display_report, audited_soap, history, key_vision_terms



In [ ]:
def format_standard_report(validated_json, warnings):
    """
    Transforms validated clinical data into a professional Markdown report.
    Prioritizes safety alerts and patient status.
    """
    # 1. Status Indicator (Visual Triage)
    status = validated_json.get("patient_status", "Unknown").upper()
    status_emoji = "🟢" if status == "STABLE" else ("🔴" if status == "CRITICAL" else "🟡")
    
    # 2. Risk Header
    risk_score = validated_json.get("risk_score", 0)
    risk_bar = "█" * risk_score + "░" * (10 - risk_score)
    
    # 3. Warning Section (Safety Gate)
    warning_section = ""
    if warnings:
        warning_list = "\n".join([f"* {w}" for w in warnings])
        warning_section = f"""
        > ### 🚨 SAFETY AUDIT ALERTS
        > {warning_list}
        > 
        > *Note: Please cross-reference findings with primary source data.*
        ---
        """
        
    # 4. Constructing the SOAP Report 
    report_md = f"""
    # {status_emoji} CLINICAL SUMMARY: {status}
    **Risk Score:** `{risk_score}/10` [{risk_bar}]
    
    {warning_section}
    
    ### 📝 SUBJECTIVE
    {validated_json.get('subjective', 'No history provided.')}
    
    ### 🔍 OBJECTIVE
    {validated_json.get('objective', 'No objective findings provided.')}
    
    ### 🧠 ASSESSMENT
    **Primary Impression:** {validated_json.get('assessment', 'Pending clinical review.')}
    
    ### 📋 PLAN
    {validated_json.get('plan', 'Awaiting clinician sign-off.')}
    
    ---
    **System Guarantees:** 🛡️ PII-Safe | ✅ Schema-Validated | ⚖️ Audit-Checked [cite: 50, 51, 52]
    """
    return report_md

## Gradio Interface